In [ ]:

from pathlib import Path
import json
import numpy as np
from PIL import Image, ImageFile
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
from tqdm import tqdm

# Fix PIL truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ============================================
# CONFIG (Kaggle-ready)
# ============================================
DATA_ROOT = Path("/kaggle/input/coco-2017-dataset/coco2017")

IMG_ROOT = {
    "train": DATA_ROOT / "train2017",
    "val": DATA_ROOT / "val2017",
}

ANN_FILE = {
    "train": DATA_ROOT / "annotations" / "instances_train2017.json",
    "val": DATA_ROOT / "annotations" / "instances_val2017.json",
}

TARGET_SIZE = (640, 640)
MIN_BOX_AREA = 32.0  # 5.7x5.7 pixels minimum

# ============================================
# STEP 1: ANOMALY DETECTION & VALIDATION
# ============================================

def validate_coco_split(split: str, save_bad_list: bool = True):
    """Find anomalies: missing images, tiny objects, invalid masks, corrupted"""
    print(f"\n🔍 Validating {split} dataset...")
    coco = COCO(str(ANN_FILE[split]))
    img_ids = coco.getImgIds()
    
    anomalies = {
        "missing_images": [],
        "tiny_boxes": [],
        "invalid_masks": [],
        "corrupted_images": [],
        "empty_annotations": []
    }
    
    valid_img_ids = []
    
    for img_id in tqdm(img_ids, desc=f"Checking {split}", leave=False):
        try:
            img_meta = coco.loadImgs([img_id])[0]
            img_path = IMG_ROOT[split] / img_meta["file_name"]
            
            # Check 1: Missing file
            if not img_path.exists():
                anomalies["missing_images"].append(img_id)
                continue
                
            # Check 2: Corrupted image
            with Image.open(img_path) as im:
                img = np.asarray(im.convert("RGB"))
                if img.size == 0 or img.shape[0] == 0:
                    anomalies["corrupted_images"].append(img_id)
                    continue
                    
            # Check 3: No annotations
            ann_ids = coco.getAnnIds(imgIds=[img_id])
            anns = coco.loadAnns(ann_ids)
            if len(anns) == 0:
                anomalies["empty_annotations"].append(img_id)
                continue
                
            # Check 4: All tiny boxes
            valid_anns = [ann for ann in anns if ann["area"] >= MIN_BOX_AREA]
            if len(valid_anns) == 0:
                anomalies["tiny_boxes"].append(img_id)
                continue
                
            # Check 5: All invalid masks
            invalid_mask_count = 0
            for ann in valid_anns:
                if "segmentation" not in ann or not ann["segmentation"]:
                    invalid_mask_count += 1
                    continue
                
                try:
                    h0, w0 = img.shape[:2]
                    if isinstance(ann["segmentation"], list):
                        rles = maskUtils.frPyObjects(ann["segmentation"], h0, w0)
                        rle = maskUtils.merge(rles)
                    else:
                        rle = ann["segmentation"]
                    mask = maskUtils.decode(rle)
                    if mask.sum() == 0:
                        invalid_mask_count += 1
                except:
                    invalid_mask_count += 1
            
            if invalid_mask_count == len(valid_anns):
                anomalies["invalid_masks"].append(img_id)
                continue
            
            valid_img_ids.append(img_id)
            
        except Exception:
            anomalies["corrupted_images"].append(img_id)
            continue
    
    # Save detailed report
    if save_bad_list:
        report = {
            "split": split,
            "total_images": len(img_ids),
            "valid_images": len(valid_img_ids),
            "percent_valid": f"{100*len(valid_img_ids)/len(img_ids):.1f}%",
            "anomalies": {k: len(v) for k, v in anomalies.items()},
            "valid_img_ids": valid_img_ids[:100],  # Sample
            "bad_details": anomalies
        }
        report_path = Path(f"/kaggle/working/anomaly_report_{split}.json")
        report_path.write_text(json.dumps(report, indent=2))
    
    print(f"✅ {split.upper()}: {len(valid_img_ids)}/{len(img_ids)} VALID ({100*len(valid_img_ids)/len(img_ids):.1f}%)")
    return valid_img_ids

# ============================================
# STEP 2: EXTRACT CLEAN MASKS ONLY
# ============================================

def extract_clean_masks(split: str, valid_img_ids: list, out_dir: Path):
    """Create multi-class mask PNGs (pixel value = COCO category ID 0-80)"""
    print(f"\n🎭 Extracting {len(valid_img_ids)} clean masks for {split}...")
    out_dir.mkdir(parents=True, exist_ok=True)
    coco = COCO(str(ANN_FILE[split]))
    
    saved_count = 0
    for img_id in tqdm(valid_img_ids, desc=f"Creating {split} masks", leave=False):
        try:
            img_meta = coco.loadImgs([img_id])[0]
            img_path = IMG_ROOT[split] / img_meta["file_name"]
            
            # Get original dimensions (no full image load)
            with Image.open(img_path) as im:
                h0, w0 = im.size[1], im.size[0]  # PIL: (W,H) → (H,W)
            
            ann_ids = coco.getAnnIds(imgIds=[img_id])
            anns = coco.loadAnns(ann_ids)
            
            # Multi-class mask: 0=background, 1-80=COCO categories
            mask_clean = np.zeros((h0, w0), dtype=np.uint16)
            
            for ann in anns:
                if ann["area"] < MIN_BOX_AREA or not ann.get("segmentation"):
                    continue
                
                cat_id = ann["category_id"]
                
                try:
                    if isinstance(ann["segmentation"], list):
                        rles = maskUtils.frPyObjects(ann["segmentation"], h0, w0)
                        rle = maskUtils.merge(rles)
                    else:
                        rle = ann["segmentation"]
                    
                    binary_mask = maskUtils.decode(rle).astype(np.uint8)
                    
                    # Overwrite with category ID (instance → semantic)
                    mask_clean[binary_mask > 0] = cat_id
                    
                except Exception:
                    continue
            
            # Save only if has valid masks
            if mask_clean.max() > 0:
                out_path = out_dir / f"{img_meta['file_name'].split('.')[0]}_mask.png"
                Image.fromarray(mask_clean).save(out_path, "PNG", compress_level=6)
                saved_count += 1
                
        except Exception as e:
            print(f"⚠️ Failed {img_meta.get('file_name', img_id)}: {e}")
            continue
    
    print(f"✅ Saved {saved_count}/{len(valid_img_ids)} clean mask PNGs")
    return saved_count

# ============================================
# STEP 3: RUN FULL PIPELINE
# ============================================

if __name__ == "__main__":
    print("🚀 COCO CLEAN MASK PIPELINE STARTED")
    
    # 1. Find valid images (filter anomalies)
    print("\n" + "="*50)
    valid_train_ids = validate_coco_split("train")
    valid_val_ids = validate_coco_split("val")
    
    # 2. Extract clean masks
    MASK_OUTPUT = Path("/kaggle/working/masks")
    train_saved = extract_clean_masks("train", valid_train_ids, MASK_OUTPUT / "train")
    val_saved = extract_clean_masks("val", valid_val_ids, MASK_OUTPUT / "val")
    
    # 3. Final summary
    print("\n" + "="*50)
    print("🎉 PIPELINE COMPLETE!")
    print(f"📁 Output: {MASK_OUTPUT}")
    print(f"   Train masks: {train_saved}")
    print(f"   Val masks:   {val_saved}")
    print(f"📊 Reports: anomaly_report_train.json, anomaly_report_val.json")
    print("\n✅ Ready for training segmentation model!")


🚀 COCO CLEAN MASK PIPELINE STARTED


🔍 Validating train dataset...
loading annotations into memory...
Done (t=14.96s)
creating index...
index created!


✅ TRAIN: 117254/118287 VALID (99.1%)

🔍 Validating val dataset...
loading annotations into memory...
Done (t=0.72s)
creating index...
index created!


✅ VAL: 4952/5000 VALID (99.0%)

🎭 Extracting 117254 clean masks for train...
loading annotations into memory...
Done (t=11.59s)
creating index...
index created!


✅ Saved 117254/117254 clean mask PNGs

🎭 Extracting 4952 clean masks for val...
loading annotations into memory...
Done (t=0.42s)
creating index...
index created!


✅ Saved 4952/4952 clean mask PNGs

🎉 PIPELINE COMPLETE!
📁 Output: /kaggle/working/masks
   Train masks: 117254
   Val masks:   4952
📊 Reports: anomaly_report_train.json, anomaly_report_val.json

✅ Ready for training segmentation model!
